# <img src="https://raw.githubusercontent.com/iterative/datachain/main/docs/assets/datachain.svg" style="height: 32px; width: 32px; vertical-align: bottom"/> Tutorial: Working with Video Datasets

This tutorial dives into techniques to manage video datasets datasets.

**📋 Topics covered:**
1. Building a Video DataChain for [AVA Actions](https://research.google.com/ava/) video dataset
2. Creating a Data Model for YOLOv11 Pose Detection projects
3. Integrating Video-Level Annotations from CSV
5. Extract and Manage Video Frames
6. Running Pose Detection with YOLOv11 and Saving to DataChain
7. Visualizing Pose Data

## Setup

To start, install the dependencies.

In [1]:
%pip install -q 'datachain[video]'

Note: you may need to restart the kernel to use updated packages.


## Get data and create DataChain

We will use **AVA Actions dataset** from [https://research.google.com/ava/download.html](https://research.google.com/ava/download.html#ava_actions_download). To download dataset, use the following script:

In [2]:
%%bash

# Download video files list
DATA_DIR="data/ava"
mkdir -p ${DATA_DIR}

wget -q https://s3.amazonaws.com/ava-dataset/annotations/ava_file_names_trainval_v2.1.txt -O ${DATA_DIR}/ava_file_names_trainval_v2.1.txt

# Download video files
VIDEOS_DIR="${DATA_DIR}/videos"
mkdir -p ${VIDEOS_DIR}

for file in $(cat ${DATA_DIR}/ava_file_names_trainval_v2.1.txt | head -3); do
    echo -n "Download '${file}'... "
    if [[ ! -f ${VIDEOS_DIR}/${file} ]]; then
        wget -q https://s3.amazonaws.com/ava-dataset/trainval/${file} -P ${VIDEOS_DIR}
    fi
    echo "OK"
done

# Download annotations
ANNOTATIONS_DIR="${DATA_DIR}/annotations"
mkdir -p ${ANNOTATIONS_DIR}

wget -q https://research.google.com/ava/download/ava_v2.2.zip -O ${DATA_DIR}/ava_v2.2.zip
unzip -qo ${DATA_DIR}/ava_v2.2.zip -d ${ANNOTATIONS_DIR}

# Prepare output directory for video clips
mkdir -p ${DATA_DIR}/clips

Download '_-Z6wFjXtGQ.mkv'... OK
Download '_145Aa_xkuE.mp4'... OK
Download '_7oWZq_s_Sk.mkv'... OK


### Indexing storage

To create a chain from a directory of files, use `DataChain.from_storage()` and point to the location of the directory:

In [3]:
from datachain import DataChain

repo_dc = DataChain.from_storage("./data/ava/videos", type="video", update=True)

Processed: 0 rows [00:00, ? rows/s]
ting file:///Users/vlad/work/iterative/datachain-examples/video/data/ava/videos: 0 objects [00:00, ? objects/s]
                                                                                                               
                                   
                                                                                                                                                                                         

Preview created chain:

In [4]:
repo_dc.show(10)

,file,file,file,file,file,file,file,file
,source,path,size,version,etag,is_latest,last_modified,location
0,file:///Users/vlad/work/iterative/datachain-ex...,9Y_l9NsnYE0.mp4,299051037,,0x1.6a6d311c00000p+30,1,2018-03-04 01:30:47+00:00,None
1,file:///Users/vlad/work/iterative/datachain-ex...,G4qq1MRXCiY.mkv,204727729,,0x1.6a6eceec00000p+30,1,2018-03-04 08:52:11+00:00,None
2,file:///Users/vlad/work/iterative/datachain-ex...,4gVsDd8PV9U.mp4,587579944,,0x1.6a6c6bb400000p+30,1,2018-03-03 22:00:13+00:00,None
3,file:///Users/vlad/work/iterative/datachain-ex...,c9pEMjPT16M.webm,340432181,,0x1.6a6de73000000p+30,1,2018-03-04 04:45:00+00:00,None
4,file:///Users/vlad/work/iterative/datachain-ex...,Gt61_Yekkgc.mp4,326660841,,0x1.6a6f0d3400000p+30,1,2018-03-04 09:58:37+00:00,None
5,file:///Users/vlad/work/iterative/datachain-ex...,-IELREHX_js.mp4,125710755,,0x1.6a6c20ec00000p+30,1,2018-03-03 20:40:27+00:00,None
6,file:///Users/vlad/work/iterative/datachain-ex...,0wBYFahr3uI.mp4,274752600,,0x1.6a6c317c00000p+30,1,2018-03-03 20:58:07+00:00,None
7,file:///Users/vlad/work/iterative/datachain-ex...,bePts02nIY8.mkv,272741181,,0x1.6a6dcdc400000p+30,1,2018-03-04 04:17:53+00:00,None
8,file:///Users/vlad/work/iterative/datachain-ex...,bSZiZ4rOC7c.mkv,220409224,,0x1.6a6dd5d000000p+30,1,2018-03-04 04:26:28+00:00,None



[Limited by 10 rows]


DataChain created a record for each file in the repository, generating a `file` signal for each file. The file signal contains subsignals with metadata about each file, like `file.name` and `file.size`. By default all `file` signals will have `File` type, by using `type="video"` param we can specify type of the files to `VideoFile`.

### Adding YouTube video ID

We will need YouTube video ID to process files. Let's create mapper to add `video_id` signal:

In [5]:
from datachain import File

def video_id(file: File) -> str:
    return file.path[:11]

repo_dc2 = repo_dc.map(video_id, output=['video_id']).save()

Preview created chain:

In [6]:
repo_dc2.show(10)

,file,file,file,file,file,file,file,file,video_id
,source,path,size,version,etag,is_latest,last_modified,location,
0,file:///Users/vlad/work/iterative/datachain-ex...,9Y_l9NsnYE0.mp4,299051037,,0x1.6a6d311c00000p+30,1,2018-03-04 01:30:47+00:00,None,9Y_l9NsnYE0
1,file:///Users/vlad/work/iterative/datachain-ex...,G4qq1MRXCiY.mkv,204727729,,0x1.6a6eceec00000p+30,1,2018-03-04 08:52:11+00:00,None,G4qq1MRXCiY
2,file:///Users/vlad/work/iterative/datachain-ex...,4gVsDd8PV9U.mp4,587579944,,0x1.6a6c6bb400000p+30,1,2018-03-03 22:00:13+00:00,None,4gVsDd8PV9U
3,file:///Users/vlad/work/iterative/datachain-ex...,c9pEMjPT16M.webm,340432181,,0x1.6a6de73000000p+30,1,2018-03-04 04:45:00+00:00,None,c9pEMjPT16M
4,file:///Users/vlad/work/iterative/datachain-ex...,Gt61_Yekkgc.mp4,326660841,,0x1.6a6f0d3400000p+30,1,2018-03-04 09:58:37+00:00,None,Gt61_Yekkgc
5,file:///Users/vlad/work/iterative/datachain-ex...,-IELREHX_js.mp4,125710755,,0x1.6a6c20ec00000p+30,1,2018-03-03 20:40:27+00:00,None,-IELREHX_js
6,file:///Users/vlad/work/iterative/datachain-ex...,0wBYFahr3uI.mp4,274752600,,0x1.6a6c317c00000p+30,1,2018-03-03 20:58:07+00:00,None,0wBYFahr3uI
7,file:///Users/vlad/work/iterative/datachain-ex...,bePts02nIY8.mkv,272741181,,0x1.6a6dcdc400000p+30,1,2018-03-04 04:17:53+00:00,None,bePts02nIY8
8,file:///Users/vlad/work/iterative/datachain-ex...,bSZiZ4rOC7c.mkv,220409224,,0x1.6a6dd5d000000p+30,1,2018-03-04 04:26:28+00:00,None,bSZiZ4rOC7c



[Limited by 10 rows]


### Create annotations DataChain

Each row in annotations CSV file contains an annotation for one person performing an action in an interval, where that annotation is associated with the middle frame. Different persons and multiple action labels are described in separate rows.

The format of a row is the following: `video_id`, `middle_frame_timestamp`, `person_box`, `action_id`, `person_id`

- `video_id`: YouTube identifier
- `middle_frame_timestamp`: in seconds from the start of the YouTube.
- `person_box`: top-left (x1, y1) and bottom-right (x2,y2) normalized with respect to frame size, where (0.0, 0.0) corresponds to the top left, and (1.0, 1.0) corresponds to bottom right.
- `action_id`: identifier of an action class, see ava_action_list_v2.2.pbtxt
- `person_id`: a unique integer allowing this box to be linked to other boxes depicting the same person in adjacent frames of this video.

In [7]:
from datachain import DataChain, DataModel

class AvaMeta(DataModel):
    video_id: str
    timestamp: int
    box_left: float
    box_top: float
    box_right: float
    box_bottom: float
    action_id: int
    person_id: int

annotations_dc = DataChain.from_csv(
    "./data/ava/annotations/ava_train_v2.2.csv",
    output=AvaMeta,
    source=False,
    settings={'prefetch': 0, 'cache': False},
).save()

Processed: 0 rows [00:00, ? rows/s]
                                   
Processed: 0 rows [00:00, ? rows/s]                                                                                                                                                      
                                   
Processed: 0 rows [00:00, ? rows/s]
sed by pyarrow: 0rows [00:00, ?rows/s]

ted: 0 rows [00:00, ? rows/s]
sed by pyarrow: 4090rows [00:00, 40899.16rows/s]

ted: 7767 rows [00:00, 77666.19 rows/s]
sed by pyarrow: 10633rows [00:00, 55326.73rows/s]

ted: 15534 rows [00:00, 71226.71 rows/s]
sed by pyarrow: 18324rows [00:00, 65181.23rows/s]

ted: 22692 rows [00:00, 60515.56 rows/s]
sed by pyarrow: 24843rows [00:00, 57972.67rows/s]

ted: 30001 rows [00:00, 62741.78 rows/s]
sed by pyarrow: 31414rows [00:00, 60554.99rows/s]

ted: 37982 rows [00:00, 68253.97 rows/s]
sed by pyarrow: 39374rows [00:00, 66704.92rows/s]

ted: 44942 rows [00:00, 51543.36 rows/s]
sed by pyarrow: 46135rows [00:00, 50569.17rows/s]


Preview:

In [8]:
annotations_dc.show(10)

,video_id,timestamp,box_left,box_top,box_right,box_bottom,action_id,person_id
0,-5KQ66BBWC4,902,0.077,0.151,0.283,0.811,9,1
1,-5KQ66BBWC4,902,0.226,0.032,0.366,0.497,12,0
2,-5KQ66BBWC4,902,0.226,0.032,0.366,0.497,17,0
3,-5KQ66BBWC4,902,0.226,0.032,0.366,0.497,80,0
4,-5KQ66BBWC4,902,0.332,0.194,0.481,0.891,80,2
5,-5KQ66BBWC4,902,0.332,0.194,0.481,0.891,9,2
6,-5KQ66BBWC4,902,0.505,0.105,0.653,0.780,9,3
7,-5KQ66BBWC4,902,0.626,0.146,0.805,0.818,9,5
8,-5KQ66BBWC4,902,0.805,0.222,0.997,1.000,80,4
9,-5KQ66BBWC4,902,0.805,0.222,0.997,1.000,9,4



[Limited by 10 rows]


Action classes:

In [9]:
actions = {
    1: {'name': 'bend/bow (at the waist)', 'cls': 'PERSON_MOVEMENT'},
    2: {'name': 'crawl', 'cls': 'PERSON_MOVEMENT'},
    3: {'name': 'crouch/kneel', 'cls': 'PERSON_MOVEMENT'},
    4: {'name': 'dance', 'cls': 'PERSON_MOVEMENT'},
    5: {'name': 'fall down', 'cls': 'PERSON_MOVEMENT'},
    6: {'name': 'get up', 'cls': 'PERSON_MOVEMENT'},
    7: {'name': 'jump/leap', 'cls': 'PERSON_MOVEMENT'},
    8: {'name': 'lie/sleep', 'cls': 'PERSON_MOVEMENT'},
    9: {'name': 'martial art', 'cls': 'PERSON_MOVEMENT'},
    10: {'name': 'run/jog', 'cls': 'PERSON_MOVEMENT'},
    11: {'name': 'sit', 'cls': 'PERSON_MOVEMENT'},
    12: {'name': 'stand', 'cls': 'PERSON_MOVEMENT'},
    13: {'name': 'swim', 'cls': 'PERSON_MOVEMENT'},
    14: {'name': 'walk', 'cls': 'PERSON_MOVEMENT'},
    15: {'name': 'answer phone', 'cls': 'OBJECT_MANIPULATION'},
    16: {'name': 'brush teeth', 'cls': 'OBJECT_MANIPULATION'},
    17: {'name': 'carry/hold (an object)', 'cls': 'OBJECT_MANIPULATION'},
    18: {'name': 'catch (an object)', 'cls': 'OBJECT_MANIPULATION'},
    19: {'name': 'chop', 'cls': 'OBJECT_MANIPULATION'},
    20: {'name': 'climb (e.g., a mountain)', 'cls': 'OBJECT_MANIPULATION'},
    21: {'name': 'clink glass', 'cls': 'OBJECT_MANIPULATION'},
    22: {'name': 'close (e.g., a door, a box)', 'cls': 'OBJECT_MANIPULATION'},
    23: {'name': 'cook', 'cls': 'OBJECT_MANIPULATION'},
    24: {'name': 'cut', 'cls': 'OBJECT_MANIPULATION'},
    25: {'name': 'dig', 'cls': 'OBJECT_MANIPULATION'},
    26: {'name': 'dress/put on clothing', 'cls': 'OBJECT_MANIPULATION'},
    27: {'name': 'drink', 'cls': 'OBJECT_MANIPULATION'},
    28: {'name': 'drive (e.g., a car, a truck)', 'cls': 'OBJECT_MANIPULATION'},
    29: {'name': 'eat', 'cls': 'OBJECT_MANIPULATION'},
    30: {'name': 'enter', 'cls': 'OBJECT_MANIPULATION'},
    31: {'name': 'exit', 'cls': 'OBJECT_MANIPULATION'},
    32: {'name': 'extract', 'cls': 'OBJECT_MANIPULATION'},
    33: {'name': 'fishing', 'cls': 'OBJECT_MANIPULATION'},
    34: {'name': 'hit (an object)', 'cls': 'OBJECT_MANIPULATION'},
    35: {'name': 'kick (an object)', 'cls': 'OBJECT_MANIPULATION'},
    36: {'name': 'lift/pick up', 'cls': 'OBJECT_MANIPULATION'},
    37: {'name': 'listen (e.g., to music)', 'cls': 'OBJECT_MANIPULATION'},
    38: {'name': 'open (e.g., a window, a car door)', 'cls': 'OBJECT_MANIPULATION'},
    39: {'name': 'paint', 'cls': 'OBJECT_MANIPULATION'},
    40: {'name': 'play board game', 'cls': 'OBJECT_MANIPULATION'},
    41: {'name': 'play musical instrument', 'cls': 'OBJECT_MANIPULATION'},
    42: {'name': 'play with pets', 'cls': 'OBJECT_MANIPULATION'},
    43: {'name': 'point to (an object)', 'cls': 'OBJECT_MANIPULATION'},
    44: {'name': 'press', 'cls': 'OBJECT_MANIPULATION'},
    45: {'name': 'pull (an object)', 'cls': 'OBJECT_MANIPULATION'},
    46: {'name': 'push (an object)', 'cls': 'OBJECT_MANIPULATION'},
    47: {'name': 'put down', 'cls': 'OBJECT_MANIPULATION'},
    48: {'name': 'read', 'cls': 'OBJECT_MANIPULATION'},
    49: {'name': 'ride (e.g., a bike, a car, a horse)', 'cls': 'OBJECT_MANIPULATION'},
    50: {'name': 'row boat', 'cls': 'OBJECT_MANIPULATION'},
    51: {'name': 'sail boat', 'cls': 'OBJECT_MANIPULATION'},
    52: {'name': 'shoot', 'cls': 'OBJECT_MANIPULATION'},
    53: {'name': 'shovel', 'cls': 'OBJECT_MANIPULATION'},
    54: {'name': 'smoke', 'cls': 'OBJECT_MANIPULATION'},
    55: {'name': 'stir', 'cls': 'OBJECT_MANIPULATION'},
    56: {'name': 'take a photo', 'cls': 'OBJECT_MANIPULATION'},
    57: {'name': 'text on/look at a cellphone', 'cls': 'OBJECT_MANIPULATION'},
    58: {'name': 'throw', 'cls': 'OBJECT_MANIPULATION'},
    59: {'name': 'touch (an object)', 'cls': 'OBJECT_MANIPULATION'},
    60: {'name': 'turn (e.g., a screwdriver)', 'cls': 'OBJECT_MANIPULATION'},
    61: {'name': 'watch (e.g., TV)', 'cls': 'OBJECT_MANIPULATION'},
    62: {'name': 'work on a computer', 'cls': 'OBJECT_MANIPULATION'},
    63: {'name': 'write', 'cls': 'OBJECT_MANIPULATION'},
    64: {'name': 'fight/hit (a person)', 'cls': 'PERSON_INTERACTION'},
    65: {'name': 'give/serve (an object) to (a person)', 'cls': 'PERSON_INTERACTION'},
    66: {'name': 'grab (a person)', 'cls': 'PERSON_INTERACTION'},
    67: {'name': 'hand clap', 'cls': 'PERSON_INTERACTION'},
    68: {'name': 'hand shake', 'cls': 'PERSON_INTERACTION'},
    69: {'name': 'hand wave', 'cls': 'PERSON_INTERACTION'},
    70: {'name': 'hug (a person)', 'cls': 'PERSON_INTERACTION'},
    71: {'name': 'kick (a person)', 'cls': 'PERSON_INTERACTION'},
    72: {'name': 'kiss (a person)', 'cls': 'PERSON_INTERACTION'},
    73: {'name': 'lift (a person)', 'cls': 'PERSON_INTERACTION'},
    74: {'name': 'listen to (a person)', 'cls': 'PERSON_INTERACTION'},
    75: {'name': 'play with kids', 'cls': 'PERSON_INTERACTION'},
    76: {'name': 'push (another person)', 'cls': 'PERSON_INTERACTION'},
    77: {'name': 'sing to (e.g., self, a person, a group)', 'cls': 'PERSON_INTERACTION'},
    78: {'name': 'take (an object) from (a person)', 'cls': 'PERSON_INTERACTION'},
    79: {'name': 'talk to (e.g., self, a person, a group)', 'cls': 'PERSON_INTERACTION'},
    80: {'name': 'watch (a person)', 'cls': 'PERSON_INTERACTION'}
}

## Video files processing

For example, let's find all frames with "**write**" action (`action_id = 63`):

In [10]:
from datachain import C

write_frames_dc = annotations_dc.filter(C("action_id") == 63)

Preview:

In [11]:
write_frames_dc.show(10)

,video_id,timestamp,box_left,box_top,box_right,box_bottom,action_id,person_id
0,-OyDO1g74vc,1342,0.009,0.005,0.630,0.995,63,62
1,-OyDO1g74vc,1343,0.003,0.014,0.610,0.995,63,62
2,-OyDO1g74vc,1344,0.000,0.002,0.599,1.000,63,62
3,-OyDO1g74vc,1345,0.009,0.001,0.601,0.992,63,62
4,-OyDO1g74vc,1346,0.008,0.002,0.609,0.993,63,62
5,-OyDO1g74vc,1347,0.005,0.004,0.615,1.000,63,62
6,-OyDO1g74vc,1348,0.000,0.003,0.578,0.995,63,62
7,-OyDO1g74vc,1349,0.002,0.008,0.593,1.000,63,62
8,-OyDO1g74vc,1350,0.005,0.009,0.570,0.998,63,62
9,-OyDO1g74vc,1351,0.023,0.000,0.603,0.981,63,62



[Limited by 10 rows]


Build clips from these frames:

In [12]:
from typing import Iterator
from datachain import C, DataModel

class ClipInterval(DataModel):
    video_id: str
    person_id: int
    start: int
    end: int
    frames: int

def process(video_id, person_id, timestamp) -> Iterator[ClipInterval]:
    yield ClipInterval(
        video_id=video_id[0],
        person_id=person_id[0],
        start=timestamp[0],
        end=timestamp[-1],
        frames=len(video_id),
    )

clips_dc = write_frames_dc.order_by("video_id", "timestamp").agg(
    process,
    output=["clip"],
    partition_by=[C("video_id"), C("person_id")],
).filter(C("clip.frames") > 1).save()

Processed: 0 rows [00:00, ? rows/s]
                                   
                                                                                                                                                                                         

Preview:

In [13]:
clips_dc.show(10)

,clip,clip,clip,clip,clip
,video_id,person_id,start,end,frames
0,-OyDO1g74vc,62,1342,1357,14
1,-XpUuIgyUHE,288,1367,1369,3
2,2FIHxnZKg6A,100,1130,1131,2
3,2fwni_Kjf2M,45,1117,1118,2
4,3_VjIRdXVdM,275,1491,1493,3
5,4ZpjKfu6Cl8,161,1295,1302,4
6,4ZpjKfu6Cl8,169,1305,1306,2
7,4ZpjKfu6Cl8,173,1313,1314,2
8,5LrOQEt_XVM,299,1671,1679,5



[Limited by 10 rows]


And create video clips from these frames using `VideoClip` model:

In [14]:
repo_clip_dc = repo_dc2.merge(
    clips_dc,
    on="video_id",
    right_on="clip.video_id",
    inner=True
)

Preview:

In [15]:
repo_clip_dc.show(10)

,file,file,file,file,file,file,file,file,video_id,clip,clip,clip,clip,clip
,source,path,size,version,etag,is_latest,last_modified,location,,video_id,person_id,start,end,frames
0,file:///Users/vlad/work/iterative/datachain-ex...,C3qk4yAMANk.mkv,492664807,,0x1.6a6de12000000p+30,1,2018-03-04 04:38:32+00:00,None,C3qk4yAMANk,C3qk4yAMANk,149,1268,1270,3
1,file:///Users/vlad/work/iterative/datachain-ex...,8VZEwOCQ8bc.mkv,406988513,,0x1.6a6ce0f000000p+30,1,2018-03-04 00:05:16+00:00,None,8VZEwOCQ8bc,8VZEwOCQ8bc,169,1771,1775,5
2,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,cWYJHb25EVs,cWYJHb25EVs,250,1609,1642,9
3,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,cWYJHb25EVs,cWYJHb25EVs,265,1646,1650,5
4,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,cWYJHb25EVs,cWYJHb25EVs,267,1657,1658,2
5,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,cWYJHb25EVs,cWYJHb25EVs,271,1660,1663,4
6,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,cWYJHb25EVs,cWYJHb25EVs,305,1781,1783,3
7,file:///Users/vlad/work/iterative/datachain-ex...,_Ca3gOdOHxU.mp4,541907613,,0x1.6a6b678000000p+30,1,2018-03-03 17:22:40+00:00,None,_Ca3gOdOHxU,_Ca3gOdOHxU,492,1195,1199,5
8,file:///Users/vlad/work/iterative/datachain-ex...,_Ca3gOdOHxU.mp4,541907613,,0x1.6a6b678000000p+30,1,2018-03-03 17:22:40+00:00,None,_Ca3gOdOHxU,_Ca3gOdOHxU,496,1201,1202,2



[Limited by 10 rows]


In [16]:
from datachain.lib.file import VideoFile, VideoClip

def process_clip(file: VideoFile, clip: ClipInterval) -> Iterator[VideoClip]:
    yield VideoClip(
        **file.model_dump(),
        start=clip.start,
        end=clip.end,
    )

video_clip_dc = repo_clip_dc.gen(process_clip, output=["file"]).save()

Processed: 0 rows [00:00, ? rows/s]
Processed: 2 rows [00:00, 13.49 rows/s]
Processed: 4 rows [00:00, 16.03 rows/s]
Processed: 6 rows [00:00, 12.68 rows/s]
Processed: 8 rows [00:00,  8.30 rows/s]
Processed: 10 rows [00:01,  6.58 rows/s]
Processed: 12 rows [00:01,  5.41 rows/s]
Processed: 14 rows [00:01,  6.65 rows/s]
Processed: 17 rows [00:02,  7.94 rows/s]
Processed: 19 rows [00:02,  8.92 rows/s]
Processed: 21 rows [00:02,  9.93 rows/s]
Processed: 23 rows [00:02, 11.32 rows/s]
Processed: 25 rows [00:02, 10.29 rows/s]
Processed: 27 rows [00:03, 10.11 rows/s]
Processed: 31 rows [00:03, 12.57 rows/s]
Processed: 33 rows [00:03, 12.87 rows/s]
Processed: 35 rows [00:03, 10.06 rows/s]
Processed: 41 rows [00:03, 14.82 rows/s]
Processed: 48 rows [00:04, 20.84 rows/s]
                                        
                                                                                                                                                                                         

Preview:

In [17]:
video_clip_dc.show(10)

,file,file,file,file,file,file,file,file,file,file
,source,path,size,version,etag,is_latest,last_modified,location,start,end
0,file:///Users/vlad/work/iterative/datachain-ex...,C3qk4yAMANk.mkv,492664807,,0x1.6a6de12000000p+30,1,2018-03-04 04:38:32+00:00,None,1268.0,1270.0
1,file:///Users/vlad/work/iterative/datachain-ex...,8VZEwOCQ8bc.mkv,406988513,,0x1.6a6ce0f000000p+30,1,2018-03-04 00:05:16+00:00,None,1771.0,1775.0
2,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,1609.0,1642.0
3,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,1646.0,1650.0
4,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,1657.0,1658.0
5,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,1660.0,1663.0
6,file:///Users/vlad/work/iterative/datachain-ex...,cWYJHb25EVs.mp4,307394571,,0x1.6a6e283c00000p+30,1,2018-03-04 05:54:23+00:00,None,1781.0,1783.0
7,file:///Users/vlad/work/iterative/datachain-ex...,_Ca3gOdOHxU.mp4,541907613,,0x1.6a6b678000000p+30,1,2018-03-03 17:22:40+00:00,None,1195.0,1199.0
8,file:///Users/vlad/work/iterative/datachain-ex...,_Ca3gOdOHxU.mp4,541907613,,0x1.6a6b678000000p+30,1,2018-03-03 17:22:40+00:00,None,1201.0,1202.0



[Limited by 10 rows]


Save video clips:

In [ ]:
import os
from datetime import datetime, timezone
from typing import Iterator
from datachain.lib.video import save_video_clip
from datachain.lib.file import VideoClip, VideoFile

def process(file: VideoClip) -> Iterator[VideoFile]:
    source = f"{os.path.dirname(file.source)}/clips"
    file_stem = file.get_file_stem()
    file_ext = file.get_file_ext()
    file_name = f"{file_stem}_{int(file.start)}_{int(file.end)}.{file_ext}"
    file_path = f"data/ava/clips/{file_name}"
    save_video_clip(file, file.start - 1, file.end + 1, file_path)
    stats = os.stat(file_path)
    yield VideoFile(
        source=source,
        path=file_name,
        size=stats.st_size,
        etag=stats.st_mtime.hex(),
        is_latest=True,
        last_modified=datetime.fromtimestamp(stats.st_mtime, tz=timezone.utc),
        location=None,
    )

clips_dc = video_clip_dc.limit(30).gen(process, output=["file"]).save()

Processed: 0 rows [00:00, ? rows/s]

                                                                                                                                                                                         

Processed: 1 rows [00:00,  3.08 rows/s]                                                                                                         | 8/75 [00:38<05:20,  4.78s/it, now=None]

                                                                                                                                                                                         

Processed: 1 rows [00:00,  3.07 rows/s]                                                                                                         | 8/75 [00:38<05:20,  4.78s/it, now=None]

Moviepy - Building video data/ava/clips/C3qk4yAMANk_1268_1270.mkv.
MoviePy - Writing audio in C3qk4yAMANk_1268_1270TEMP_MPY_wvf_snd.mp4



nk:   0%|                                                                                                                                            | 0/89 [00:00<?, ?it/s, now=None]
                                                                                                                                                                                      

                                                                                                                                                                                         

Processed: 1 rows [00:00,  2.11 rows/s]                                                                                                         | 8/75 [00:38<05:21,  4.80s/it, now=None]

                                                                                                                                                                                         

Processed: 1 rows [00:00,  2.10 rows/s]                                

MoviePy - Done.
Moviepy - Writing video data/ava/clips/C3qk4yAMANk_1268_1270.mkv




 94%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎       | 113/120 [00:00<00:00, 358.09it/s, now=None]
                                                                                                                                                                                      

                                                                                                                                                                                         

Processed: 1 rows [00:00,  1.12 rows/s]                                                                                                         | 8/75 [00:38<05:25,  4.86s/it, now=None]

                                                                                                                                                                                         

Processed: 1 rows [00:00,  1.11 rows/s]                                

...




0%|                                                                                                                                           | 0/199 [00:00<?, ?it/s, now=None]


5%|██████████████████████████████████████████████████████████████████████▊                                                         | 110/199 [00:00<00:00, 946.44it/s, now=None]


                                                                                                                                                                                

                                                                                                                                                                                   
                                        

%|██████████████▌                                                                                                                         | 8/75 [00:46<06:28,  5.80s/it, now=None]
Processed: 10 rows [00:08,  1.23 rows/s]

                 

MoviePy - Done.




%|██████████████▌                                                                                                                         | 8/75 [00:46<06:28,  5.80s/it, now=None]
Processed: 10 rows [00:08,  1.23 rows/s]

Moviepy - Writing video data/ava/clips/4ZpjKfu6Cl8_1295_1302.mkv






                                                                                                                                              | 0/270 [00:00<?, ?it/s, now=None]


████████████████████▋                                                                                                               | 44/270 [00:00<00:00, 439.12it/s, now=None]


██████████████████████████████████████████▎                                                                                         | 88/270 [00:00<00:00, 344.40it/s, now=None]

Preview:

In [ ]:
clips_dc.show(10)